# Joint Kinematics: Ensemble Average Across Gait Cycles

This notebook is **Stage 2** of my joint kinematics analysis pipeline. It takes the per-joint gait-cycle CSVs produced by Stage 1 (`vicon_joint_kinematics_peak_rom.ipynb`), combines the left and right sides for each joint, and computes the ensemble mean and standard deviation across all gait cycles within a subject.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived. Reflects my analytical approach during PhD dissertation research.

## Input dependencies

This notebook expects six per-joint gait-cycle CSV files per subject:

- `{Subject}_LAGC_data.csv`, `{Subject}_RAGC_data.csv` — left/right ankle
- `{Subject}_LHGC_data.csv`, `{Subject}_RHGC_data.csv` — left/right hip
- `{Subject}_LKGC_data.csv`, `{Subject}_RKGC_data.csv` — left/right knee

Each file has gait cycles as columns and time points within the cycle as rows.

## Pipeline context

This is the **second stage** in a multi-stage joint kinematics workflow:

1. `vicon_joint_kinematics_peak_rom.ipynb` — per-trial gait-cycle segmentation and outlier removal
2. **This notebook** — pool left and right cycles per joint, compute ensemble mean and SD
3. `joint_kinematics_group_statistics.ipynb` — cross-subject outlier removal and group-level comparison statistics

## What this notebook computes

For each of the three joints (ankle, hip, knee):

- Pool all gait cycles from both sides into one wide dataframe
- Compute the **ensemble mean** — the average joint angle at each percent of the gait cycle, across all cycles
- Compute the **ensemble standard deviation** — the cycle-to-cycle variability at each percent of the cycle

The mean and SD curves describe the typical joint movement pattern for the subject and how consistent it is across cycles.

---


## 1. Setup and Data Import

Import the six gait-cycle CSV files (one per joint, per side) for the subject.

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import os,sys
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [ ]:
# Check current working directory
os.getcwd()

In [ ]:
# File import by filename

Subject = input("Subject: ")

LAGC = pd.read_csv(Subject + '_LAGC_data.csv')
LAGC = LAGC.drop("Unnamed: 0", axis=1)
RAGC = pd.read_csv(Subject + '_RAGC_data.csv')
RAGC = RAGC.drop("Unnamed: 0", axis=1)
LHGC = pd.read_csv(Subject + '_LHGC_data.csv')
LHGC = LHGC.drop("Unnamed: 0", axis=1)
RHGC = pd.read_csv(Subject + '_RHGC_data.csv')
RHGC = RHGC.drop("Unnamed: 0", axis=1)
LKGC = pd.read_csv(Subject + '_LKGC_data.csv')
LKGC = LKGC.drop("Unnamed: 0", axis=1)
RKGC = pd.read_csv(Subject + '_RKGC_data.csv')
RKGC = RKGC.drop("Unnamed: 0", axis=1)

## 2. Combine Left and Right Sides per Joint

Pool the left and right sides into one wide dataframe per joint. This treats all gait cycles (regardless of side) as samples from the same distribution, which is appropriate when left and right are expected to behave symmetrically and the analysis target is overall joint behavior rather than asymmetry.

In [ ]:
# Concate Left and Right Sides

ANK = pd.concat([LAGC, RAGC], axis=1)
HIP = pd.concat([LHGC, RHGC], axis=1)
KNE = pd.concat([LKGC, RKGC], axis=1)

Rename the columns to a uniform `col1, col2, ...` pattern so the column labels do not carry trial-specific information into downstream processing.

In [ ]:
# Set up Column Names

df_names = [ANK, HIP, KNE]

for df in df_names:
    original_columns = df.columns.tolist() 
    df.columns = [f"col{i+1}" for i in range(len(df.columns))]
    

## 3. Ensemble Mean and Standard Deviation

Transpose each dataframe so that rows are gait cycles and columns are time points within the cycle. Then `mean()` and `std()` produce the ensemble curves (mean and SD as functions of percent gait cycle).

In [ ]:
# Transpose Data Frame for Ensemble Calculation

ANK_transposed = ANK.T  
HIP_transposed = HIP.T  
KNE_transposed = KNE.T 

In [ ]:
# Calculate Ensemble mean and SD
ANK_ensemble_mean = ANK_transposed.mean()
ANK_ensemble_std = ANK_transposed.std()
HIP_ensemble_mean = HIP_transposed.mean()
HIP_ensemble_std = HIP_transposed.std()
KNE_ensemble_mean = KNE_transposed.mean()
KNE_ensemble_std = KNE_transposed.std()

## 4. Save Ensemble Results

Append the per-subject ensemble mean and SD to cumulative CSVs (one per joint per statistic). These cumulative files build up across subjects and are what Stage 3 (`joint_kinematics_group_statistics.ipynb`) uses as input.

In [ ]:
# Save Ensemble mean and SD data in CSV (by Joint)
new_data = ANK_ensemble_mean
csv_file = 'ANK_EM2.csv'

if not os.path.exists(csv_file):
    new_data.to_csv(csv_file, index=Subject, mode='w')
else:
    existing_data = pd.read_csv(csv_file)
    updated_data = pd.concat([existing_data.iloc[:,1:], new_data], axis=1)
    updated_data.to_csv(csv_file, index=Subject)

In [ ]:
# Save Ensemble mean and SD data in CSV (by Joint)
new_data = HIP_ensemble_mean
csv_file = 'HIP_EM2.csv'

if not os.path.exists(csv_file):
    new_data.to_csv(csv_file, index=Subject, mode='w')
else:
    existing_data = pd.read_csv(csv_file)
    updated_data = pd.concat([existing_data.iloc[:,1:], new_data], axis=1)
    updated_data.to_csv(csv_file, index=Subject)

In [ ]:
# Save Ensemble mean and SD data in CSV (by Joint)
new_data = KNE_ensemble_mean
csv_file = 'KNE_EM2.csv'

if not os.path.exists(csv_file):
    new_data.to_csv(csv_file, index=Subject, mode='w')
else:
    existing_data = pd.read_csv(csv_file)
    updated_data = pd.concat([existing_data.iloc[:,1:], new_data], axis=1)
    updated_data.to_csv(csv_file, index=Subject)

In [ ]:
# Save Ensemble mean and SD data in CSV (by Joint)
new_data = ANK_ensemble_std
csv_file = 'ANK_SD2.csv'

if not os.path.exists(csv_file):
    new_data.to_csv(csv_file, index=Subject, mode='w')
else:
    existing_data = pd.read_csv(csv_file)
    updated_data = pd.concat([existing_data.iloc[:,1:], new_data], axis=1)
    updated_data.to_csv(csv_file, index=Subject)

In [ ]:
# Save Ensemble mean and SD data in CSV (by Joint)
new_data = HIP_ensemble_std
csv_file = 'HIP_SD2.csv'

if not os.path.exists(csv_file):
    new_data.to_csv(csv_file, index=Subject, mode='w')
else:
    existing_data = pd.read_csv(csv_file)
    updated_data = pd.concat([existing_data.iloc[:,1:], new_data], axis=1)
    updated_data.to_csv(csv_file, index=Subject)

In [ ]:
# Save Ensemble mean and SD data in CSV (by Joint)
new_data = KNE_ensemble_std
csv_file = 'KNE_SD2.csv'

if not os.path.exists(csv_file):
    new_data.to_csv(csv_file, index=Subject, mode='w')
else:
    existing_data = pd.read_csv(csv_file)
    updated_data = pd.concat([existing_data.iloc[:,1:], new_data], axis=1)
    updated_data.to_csv(csv_file, index=Subject)

## 5. Visualize Ensemble Curves

Plot the mean joint angle through the gait cycle with a shaded ±1 SD band, one figure per joint. These plots are the per-subject ensemble curves — the typical movement pattern with cycle-to-cycle variability visible as the band thickness.

In [ ]:
# Plot results (Ankle)
plt.figure(figsize=(8, 5))
plt.plot(ANK_ensemble_mean.index, ANK_ensemble_mean, label="Mean Joint Angle", color="b")
plt.fill_between(ANK_ensemble_mean.index, 
                 ANK_ensemble_mean - ANK_ensemble_std, 
                 ANK_ensemble_mean + ANK_ensemble_std, 
                 color="b", alpha=0.2, label="Std Dev")
plt.xlabel("Normalized Time")
plt.ylabel("Ankle Angle (degrees)")
plt.legend()
plt.title("Ensemble Averaging of Joint Kinematics")
plt.show()

In [ ]:
# Plot results
plt.figure(figsize=(8, 5))
plt.plot(HIP_ensemble_mean.index, HIP_ensemble_mean, label="Mean Joint Angle", color="b")
plt.fill_between(HIP_ensemble_mean.index, 
                 HIP_ensemble_mean - HIP_ensemble_std, 
                 HIP_ensemble_mean + HIP_ensemble_std, 
                 color="b", alpha=0.2, label="Std Dev")
plt.xlabel("Normalized Time")
plt.ylabel("Hip Angle (degrees)")
plt.legend()
plt.title("Ensemble Averaging of Joint Kinematics")
plt.show()

In [ ]:
# Plot results
plt.figure(figsize=(8, 5))
plt.plot(KNE_ensemble_mean.index, KNE_ensemble_mean, label="Mean Joint Angle", color="b")
plt.fill_between(KNE_ensemble_mean.index, 
                 KNE_ensemble_mean - KNE_ensemble_std, 
                 KNE_ensemble_mean + KNE_ensemble_std, 
                 color="b", alpha=0.2, label="Std Dev")
plt.xlabel("Normalized Time")
plt.ylabel("Hip Angle (degrees)")
plt.legend()
plt.title("Ensemble Averaging of Joint Kinematics")
plt.show()